In [1]:
# this notebook connects with a remote model and also 
# defines custom tools

In [1]:
from smolagents import CodeAgent, tool
from smolagents.models import InferenceClientModel
import os
import datetime
import requests
import pytz
import yaml
from tools.final_answer import FinalAnswerTool

/Users/raeez/.pyenv/versions/jupyter-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
HF_TOKEN = os.getenv("HF_TOKEN")

In [3]:
model = InferenceClientModel(
max_tokens=2096,
temperature=0.5,
token=HF_TOKEN,
model_id='Qwen/Qwen2.5-Coder-32B-Instruct',# it is possible that this model may be overloaded
custom_role_conversions=None,
)


In [4]:
final_answer = FinalAnswerTool()

In [5]:
@tool
def repo_owner_name()-> str: #it's import to specify the return type
    """A simple tool that returns the full name of owner of this repo
    Args:
        arg1: the first argument
        arg2: the second argument
    """
    return "Mohammad Raeez"

In [6]:
@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """A tool that fetches the current local time in a specified timezone.
    Args:
        timezone: A string representing a valid timezone (e.g., 'America/New_York').
    """
    try:
        # Create timezone object
        tz = pytz.timezone(timezone)
        # Get current time in that timezone
        local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        return f"The current local time in {timezone} is: {local_time}"
    except Exception as e:
        return f"Error fetching time for timezone '{timezone}': {str(e)}"

In [7]:
with open("prompts.yaml", 'r') as stream:
    prompt_templates = yaml.safe_load(stream)

In [8]:
agent = CodeAgent(
    model=model,
    tools=[final_answer, repo_owner_name,get_current_time_in_timezone], ## add your tools here (don't remove final answer)
    max_steps=6,
    verbosity_level=2,
    planning_interval=None,
    name=None,
    description=None,
    # prompt_templates=prompt_templates
)

In [9]:
r = agent.run("What is the name of owner")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the name of owner                                                                                       │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To find the name of the owner, I will use the `repo_owner_name` tool which is designed to return the full 
name of the owner of the repository.                                                                               
                                                                                                                   
<code>                                                                                                             
owner_name = repo_owner_name()                                                                                     
print(owner_name)                                                                                                  
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  owner_name = repo_owner_name()                                                                                   
  print(owner_name)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Mohammad Raeez

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: It appears that the `repo_owner_name` tool returned the name "Mohammad Raeez" but the print statement did 
not capture it properly. I will call the `repo_owner_name` tool again and directly use its output in the           
`final_answer` function.                                                                                           
                                                                                                                   
<code>                                                                                                             
owner_name = repo_owner_name()                                                                                     
final_answer(owner_name)                                                                                           
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  owner_name = repo_owner_name()                                                                                   
  final_answer(owner_name)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Mohammad Raeezi

[Step 2: Duration 4.24 seconds| Input tokens: 4,348 | Output tokens: 126]

In [10]:
r

'Mohammad Raeezi'

In [ ]:
t = agent.run("What is the time now in London")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the time now in London                                                                                  │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To find the current time in London, I will use the `get_current_time_in_timezone` tool with the           
appropriate timezone argument for London.                                                                          
                                                                                                                   
<code>                                                                                                             
london_time = get_current_time_in_timezone(timezone='Europe/London')                                               
print(london_time)                                                                                                 
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  london_time = get_current_time_in_timezone(timezone='Europe/London')                                             
  print(london_time)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
The current local time in Europe/London is: 2026-02-23 15:48:51

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: It appears that the `get_current_time_in_timezone` function is returning the time directly without needing
further splitting. I'll simply capture and print the result again to ensure I get the correct time value.          
                                                                                                                   
<code>                                                                                                             
london_time = get_current_time_in_timezone(timezone='Europe/London')                                               
print(london_time)                                                                                                 
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  london_time = get_current_time_in_timezone(timezone='Europe/London')                                             
  print(london_time)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
The current local time in Europe/London is: 2026-02-23 15:48:59

Out: None

[Step 3: Duration 4.07 seconds| Input tokens: 6,864 | Output tokens: 214]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
t